# Task 1.2: Key Assumptions
## Paper: Training and Testing Low-degree Polynomial Data Mappings via Linear SVM
**Authors:** Yin-Wen Chang, Cho-Jui Hsieh, Kai-Wei Chang, Michael Ringgaard, Chih-Jen Lin (2010)

---

## Assumption 1: Data Sparsity

**Assumption:** The input data must be sparse — most entries in each feature vector $\mathbf{x}_i$ are zero, so the average number of nonzero features $\bar{n}$ is much smaller than the total dimension $n$.

**Why the method needs it:** The whole speed advantage comes from the mapped features being sparse too. When $\bar{n} \ll n$, the mapped dimension $\hat{n} \approx \bar{n}^2/2$ stays manageable (Equation 10). If data is dense, $\hat{n}$ blows up to roughly $n^2/2$, and computing / storing the mapped features becomes more expensive than just running kernel SVM directly.

**Violation scenario:** Image datasets like MNIST, where almost every pixel has a nonzero value ($\bar{n} \approx n = 784$). The mapped features would have $\sim 307{,}000$ nonzeros per instance, making the method slow and memory-hungry — the opposite of what it's designed for.

**Paper reference:** Section 3.3, Equation (10). All benchmark datasets in Table 3 are sparse. The paper notes explicitly that the cost advantage holds when "data are sparse ($\bar{n} \ll n$)."

---

## Assumption 2: Low-Degree Polynomial Sufficiency

**Assumption:** A low-degree polynomial (degree 2 or 3) captures enough nonlinear structure in the data to achieve good classification accuracy.

**Why the method needs it:** The explicit mapping is only practical at low degrees because the feature space grows as $\binom{n+d}{d}$. At degree 4 or higher, even for moderate $n$, the mapped dimension becomes intractable. If the data actually needs complex, high-order interactions (like those captured by an RBF kernel in infinite-dimensional space), a degree-2 mapping simply won't fit the decision boundary well enough.

**Violation scenario:** A dataset with highly non-linear, concentric decision boundaries — the kind where an RBF kernel significantly outperforms any fixed-degree polynomial. The paper's own results show this: on covtype (Table 5), degree-2 mapping gets 80.90% accuracy while RBF kernel SVM reaches 84.76%.

**Paper reference:** Section 3.5 discusses extending to higher degrees and approximating RBF. Table 5 compares polynomial mapping accuracy against RBF kernel results across datasets.

---

## Assumption 3: Feature Independence for Sparsity Estimation

**Assumption:** The nonzero patterns of different features are roughly independent — whether feature $i$ is nonzero doesn't strongly predict whether feature $j$ is nonzero.

**Why the method needs it:** The paper estimates how many entries of $\mathbf{w}$ will be nonzero (Equation 11) by assuming the probability that features $i$ and $j$ co-occur equals the product of their individual nonzero rates. This estimate is used to predict memory usage. If features are heavily correlated (always appearing together) or mutually exclusive (never co-occurring), the estimate can be quite far off, potentially leading to unexpected memory issues.

**Violation scenario:** One-hot encoded categorical features, where within each category group, exactly one feature is active at a time. The features within a group are mutually exclusive, not independent. The independence assumption would predict cross-term weights between features in the same group, but those terms can never be nonzero in practice, leading to an overestimate of $\text{nnz}(\mathbf{w})$.

**Paper reference:** Section 3.4, Equation (11). The paper states it assumes features are independent when deriving the sparsity estimate for $\mathbf{w}$.